# Nigeria Insurance Compliance Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/comply54/comply54/blob/main/notebooks/03_nigeria_insurance.ipynb)

This notebook demonstrates how to evaluate insurance AI-agent actions with `comply54`.

The `NigeriaInsuranceCompliance` sector pack combines regulatory and safety controls covering:

- Nigerian Insurance Industry Reform Act 2025 (NIIRA 2025)
- NAICOM Operational Guidelines 2021
- NAICOM Market Conduct Guidelines 2023
- Nigeria Data Protection Act 2023
- MLPPA 2022 / NFIU AML Guidelines
- OWASP Agentic AI safety controls

We'll evaluate claims-processing and fraud-detection actions to see how comply54 allows, audits, escalates, or denies actions before execution.

In [ ]:
%pip install comply54

In [ ]:
from comply54.sectors import NigeriaInsuranceCompliance

compliance = NigeriaInsuranceCompliance()

print(compliance.name)
print("Jurisdictions:", compliance.jurisdictions)
print("Regulations:")
for regulation in compliance.regulations:
    print(f"  - {regulation}")

In [ ]:
def show_result(result):
    print(f"Overall decision: {result.overall.upper()}")
    print(f"Blocked: {result.blocked}")
    print(f"Audit ID: {result.audit_id}")
    print(f"Evaluated at: {result.evaluated_at}")

    notable_decisions = [
        decision for decision in result.decisions
        if decision.action != "allow"
    ]

    if notable_decisions:
        print("\nPolicy decisions:")
        for decision in notable_decisions:
            print(f"\n  Pack: {decision.pack}")
            print(f"  Regulation: {decision.regulation}")
            print(f"  Decision: {decision.action.upper()}")

            for message in decision.messages:
                print(f"  Message: {message}")

            if decision.citations:
                print("  Citations:")
                for citation in decision.citations:
                    print(
                        f"    - {citation.document}, "
                        f"{citation.section} ({citation.authority}, {citation.year})"
                    )
    else:
        print("\nNo non-allow policy decisions were triggered.")

## Scenario 1 — Read-only claim status

The AI assistant retrieves the current status of an existing insurance claim without modifying the claim or making a financial decision.

No compliance restriction is triggered, so comply54 returns `allow`.

In [ ]:
claim_status = compliance.check(
    action="get_claim_status",
    params={
        "claim_id": "CLM-00123456",
    },
    context={},
)

show_result(claim_status)

## Scenario 2 — Automated claim denial without a human adjuster

The AI attempts to deny a ₦1.5 million insurance claim without a human adjuster assigned to the case.

Because claim denial requires appropriate human involvement under the insurance compliance controls, comply54 denies the action.

In [ ]:
automated_denial = compliance.check(
    action="deny_claim",
    params={
        "claim_id": "CLM-00234567",
        "claim_amount": 1_500_000,
        "denial_reason": "Insufficient supporting documentation",
    },
    context={
        "human_adjuster_assigned": False,
    },
)

show_result(automated_denial)

## Scenario 3 — Approving a small claim with an adjuster assigned

A human adjuster is assigned and the AI proposes approval of an ₦800,000 claim after the supporting documentation has been verified.

The action is permitted to proceed, while comply54 records an `audit` decision for compliance traceability.

In [ ]:
small_claim = compliance.check(
    action="approve_claim",
    params={
        "claim_id": "CLM-00345678",
        "claim_amount": 800_000,
        "adjuster_notes": "Documentation verified",
    },
    context={
        "human_adjuster_assigned": True,
        "senior_approval": False,
    },
)

show_result(small_claim)

## Scenario 4 — High fraud score requiring investigation

The AI flags a claim with a fraud score of `0.78` after detecting suspicious repeat-claim activity.

Rather than making an autonomous fraud determination, comply54 escalates the action for human investigation and review.

In [ ]:
fraud_flag = compliance.check(
    action="flag_fraud",
    params={
        "claim_id": "CLM-00700007",
        "fraud_indicators": "Same vehicle reported stolen twice in 14 months",
        "fraud_score": 0.78,
    },
    context={
        "human_adjuster_assigned": True,
    },
)

show_result(fraud_flag)

## What happens next — LangGraph integration

Direct compliance checks are useful for evaluating individual insurance actions, but an AI-agent workflow can enforce these checks automatically before a tool executes.

comply54 provides `Comply54Guard`, which can sit between the agent and insurance tools in a LangGraph workflow:

```python
from comply54.sectors import NigeriaInsuranceCompliance
from comply54.langchain import Comply54Guard, comply54_route
from langgraph.graph import StateGraph
from langgraph.prebuilt import ToolNode

compliance = NigeriaInsuranceCompliance()

guard = Comply54Guard(
    compliance=compliance,
    context={
        "human_adjuster_assigned": True,
        "senior_approval": False,
        "human_underwriter": False,
    },
)

graph = StateGraph(InsuranceAgentState)

graph.add_node("agent", call_model)
graph.add_node("comply54_guard", guard)
graph.add_node("tools", ToolNode(insurance_tools))

graph.add_conditional_edges(
    "comply54_guard",
    comply54_route,
    {"tools": "tools", "agent": "agent"},
)
```

The guard evaluates proposed insurance tool calls before they reach the tool node. This allows actions such as unauthorized claim denial or high-risk fraud decisions to be denied or escalated before execution.

> **Note:** The executable examples in this notebook require only `comply54`. Running the LangGraph integration itself requires the optional LangGraph dependencies (`comply54[langgraph]`). The snippet above is an integration sketch; `InsuranceAgentState`, `call_model`, and `insurance_tools` are application-specific components.